# BTC/USDT · PPO 强化学习训练（RLlib 并行版）

本 Notebook 以 `crypto_ppo_training.ipynb`（SB3）为基准，使用 **Ray RLlib** 替代 Stable Baselines 3，
利用 **10 个 CPU 核心**（9 个 rollout worker + 1 个 learner）进行并行训练，对比训练速度与效果。

| 项目 | SB3 版本 | RLlib 版本 |
|------|---------|----------|
| 框架 | Stable Baselines 3 | Ray RLlib 2.x |
| 并行度 | 单进程 | 9 rollout workers + 1 learner |
| 总步数 | 300k | 300k（等效对比）|
| CPU 使用 | 1 核 | 10 核 |
| SB3 基准耗时 | ~9 分钟 | — |

**观测特征**：EMA 偏差率、布林带位置、成交量比率、RSI、MACD  
**奖励函数**：步骤收益 − 回撤惩罚  
**数据来源**：PostgreSQL (OHLCV) + Binance/数据库 (爆仓)

## 1. 配置

In [1]:
# ── 数据参数（与 crypto_kline_analysis 保持一致）────────────────
SYMBOL      = "BTC/USDT"
START_DATE  = "2026-04-09 00:00:00"
END_DATE    = "2026-04-13 00:00:00"
TIMEZONE    = "Asia/Shanghai"
DB_TABLE    = "public.crypto_kline_binance"
ONLY_CLOSED = True

# ── RL 超参数（与 SB3 版本保持一致）────────────────────────────
WINDOW_SIZE     = 30        # 滑动观测窗口（分钟数）
INITIAL_BALANCE = 10_000.0  # 初始资金（USD）
COMMISSION      = 0.001     # 单向手续费率
PENALTY_WEIGHT  = 0.05      # 回撤惩罚系数
TRAIN_RATIO     = 0.8       # 训练集比例

# ── RLlib PPO 超参数 ─────────────────────────────────────────────
NUM_WORKERS     = 9         # rollout workers（9 核采样）
NUM_LEARNERS    = 1         # learner（1 核更新）
TOTAL_TIMESTEPS = 1_000_000   # 与 SB3 等量对比
PPO_KWARGS = dict(
    lr            = 3e-4,
    train_batch_size_per_learner = 2048,  # 每个 learner 的 batch
    minibatch_size = 64,
    num_epochs     = 10,
    gamma          = 0.99,
    lambda_        = 0.95,
    clip_param     = 0.2,
    entropy_coeff  = 0.005,
    vf_loss_coeff  = 0.5,
    grad_clip      = 0.5,
)
RUN_NAME   = f"{SYMBOL.replace('/','')}__rllib_ppo"
# 使用绝对路径（save_to_path 需要绝对路径，不支持相对路径）
from pathlib import Path as _Path
MODEL_DIR  = str(_Path('../models/saved').resolve())
LOG_DIR    = str(_Path('../models/logs').resolve())
# SB3 基准（用于速度对比）
SB3_BASELINE_SECONDS = 540  # ~9 分钟
SB3_BASELINE_STEPS   = 300_000
# ────────────────────────────────────────────────────────────────

## 2. 数据加载与特征工程

### 2.1 OHLCV 加载

In [2]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import time
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK JP', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

from utils.db import read_ohlcv

tz = TIMEZONE
start_utc_iso = pd.Timestamp(START_DATE, tz=tz).tz_convert('UTC').isoformat()
end_utc_iso   = pd.Timestamp(END_DATE,   tz=tz).tz_convert('UTC').isoformat()

# 保留 tz-aware Timestamp 供爆仓查询使用
start_utc = pd.Timestamp(START_DATE, tz=tz).tz_convert('UTC')
end_utc   = pd.Timestamp(END_DATE,   tz=tz).tz_convert('UTC')

df_raw = read_ohlcv(SYMBOL, start=start_utc_iso, end=end_utc_iso,
                    table=DB_TABLE, only_closed=ONLY_CLOSED)

ts = df_raw['timestamp']
if ts.dt.tz is None:
    ts = ts.dt.tz_localize('UTC')
df_raw['timestamp'] = ts.dt.tz_convert(tz)

print(f'OHLCV 行数: {len(df_raw)}  ({df_raw["timestamp"].iloc[0]}  →  {df_raw["timestamp"].iloc[-1]})')
df_raw.head(3)

OHLCV 行数: 5760  (2026-04-09 00:00:00+08:00  →  2026-04-12 23:59:00+08:00)


,timestamp,open,high,low,close,volume
0,2026-04-09 00:00:00+08:00,71309.99,71341.79,71292.31,71329.98,14.22841
1,2026-04-09 00:01:00+08:00,71329.97,71350.00,71303.00,71310.04,4.60235
2,2026-04-09 00:02:00+08:00,71310.05,71365.50,71310.05,71365.50,17.65297


### 2.2 指标计算与特征合并

In [3]:
d = df_raw.copy()

# ── EMA ──────────────────────────────────────────────────────────
d['ema9']  = d['close'].ewm(span=9,  adjust=False).mean()
d['ema21'] = d['close'].ewm(span=21, adjust=False).mean()
d['ema55'] = d['close'].ewm(span=55, adjust=False).mean()

# ── Bollinger Band (20, 2σ) ───────────────────────────────────────
d['bb_mid']   = d['close'].rolling(20).mean()
d['bb_std']   = d['close'].rolling(20).std()
d['bb_upper'] = d['bb_mid'] + 2 * d['bb_std']
d['bb_lower'] = d['bb_mid'] - 2 * d['bb_std']

# ── RSI (14) ─────────────────────────────────────────────────────
delta = d['close'].diff()
d['rsi'] = 100 - 100 / (1 + delta.clip(lower=0).rolling(14).mean()
                           / (-delta.clip(upper=0)).rolling(14).mean().replace(0, np.nan))

# ── MACD (12, 26, 9) ─────────────────────────────────────────────
ema12 = d['close'].ewm(span=12, adjust=False).mean()
ema26 = d['close'].ewm(span=26, adjust=False).mean()
d['macd']        = ema12 - ema26
d['macd_signal'] = d['macd'].ewm(span=9, adjust=False).mean()
d['macd_hist']   = d['macd'] - d['macd_signal']

# ── 成交量均线 ────────────────────────────────────────────────────
d['vol_ma20'] = d['volume'].rolling(20).mean()

d = d.dropna().reset_index(drop=True)
print(f'有效行数: {len(d)}')
d.head(3)

有效行数: 5741


,timestamp,open,high,low,close,volume,ema9,ema21,ema55,bb_mid,bb_std,bb_upper,bb_lower,rsi,macd,macd_signal,macd_hist,vol_ma20
0,2026-04-09 00:19:00+08:00,71359.82,71404.79,71327.20,71377.39,3.31091,71336.384152,71337.884195,71336.638228,71345.3940,42.378752,71430.151503,71260.636497,47.736043,-1.520412,-1.895865,0.375453,8.574592
1,2026-04-09 00:20:00+08:00,71377.40,71413.08,71364.96,71364.97,5.57231,71342.101321,71340.346541,71337.650077,71347.1435,42.431137,71432.005774,71262.281226,45.397481,0.860212,-1.344649,2.204861,8.141787
2,2026-04-09 00:21:00+08:00,71364.97,71398.76,71343.84,71398.76,3.17752,71353.433057,71345.656855,71339.832575,71351.5795,42.982039,71437.543578,71265.615422,47.909962,5.411069,0.006494,5.404575,8.070545


### 2.3 归一化特征矩阵

In [4]:
feat = d.copy()

feat['ret_1m']       = feat['close'].pct_change().fillna(0).clip(-0.1, 0.1)
feat['hl_ratio']     = (feat['high'] - feat['low']) / (feat['close'] + 1e-8)
feat['body_ratio']   = (feat['close'] - feat['open']) / (feat['high'] - feat['low'] + 1e-8)
feat['ema9_r']       = (feat['ema9']  - feat['close']) / (feat['close'] + 1e-8)
feat['ema21_r']      = (feat['ema21'] - feat['close']) / (feat['close'] + 1e-8)
feat['ema55_r']      = (feat['ema55'] - feat['close']) / (feat['close'] + 1e-8)
feat['bb_pos']       = ((feat['close'] - feat['bb_lower'])
                        / (feat['bb_upper'] - feat['bb_lower'] + 1e-8)).clip(0, 2)
feat['vol_ratio']    = np.log1p(feat['volume'] / (feat['vol_ma20'] + 1e-8))
feat['rsi_norm']     = feat['rsi'] / 100
feat['macd_norm']    = feat['macd'] / (feat['close'] + 1e-8)
feat['macd_hist_norm']= feat['macd_hist'] / (feat['close'] + 1e-8)

FEATURE_COLS = [
    'ret_1m', 'hl_ratio', 'body_ratio',
    'ema9_r', 'ema21_r', 'ema55_r', 'bb_pos',
    'vol_ratio',
    'rsi_norm', 'macd_norm', 'macd_hist_norm',
]

feat_matrix = feat[FEATURE_COLS].ffill().fillna(0).values.astype(np.float32)
close_arr   = feat['close'].values.astype(np.float32)

print(f'特征矩阵形状 : {feat_matrix.shape}  ({len(FEATURE_COLS)} 维 × {len(feat)} 步)')
print(f'特征列       : {FEATURE_COLS}')

特征矩阵形状 : (5741, 11)  (11 维 × 5741 步)
特征列       : ['ret_1m', 'hl_ratio', 'body_ratio', 'ema9_r', 'ema21_r', 'ema55_r', 'bb_pos', 'vol_ratio', 'rsi_norm', 'macd_norm', 'macd_hist_norm']


## 3. 交易环境（写入文件供 Ray Worker 导入）

Ray 的 rollout worker 运行在独立子进程中，无法直接访问 notebook 局部变量，
因此需将环境类写入 `/tmp/crypto_ppo_env.py`，让 worker 通过 `import` 加载。

环境与 SB3 版本完全相同：
- **观测空间**：`(WINDOW_SIZE × 11 + 2,)` 展平向量
- **动作空间**：Discrete(3) — 0=持仓, 1=全仓买入, 2=全仓卖出
- **奖励**：`step_return − PENALTY_WEIGHT × drawdown`

In [5]:
import textwrap
from pathlib import Path

# 将环境类写入 /tmp 供 Ray worker 子进程 import
ENV_FILE = Path('/tmp/crypto_ppo_env.py')

env_source = textwrap.dedent('''
    import numpy as np
    import gymnasium as gym
    from gymnasium import spaces


    class CryptoPPOEnv(gym.Env):
        metadata = {"render_modes": []}

        def __init__(self, env_config=None):
            super().__init__()
            cfg              = env_config or {}
            import ray
            feat_ref  = cfg['features_ref']
            price_ref = cfg['prices_ref']
            self.features        = ray.get(feat_ref)
            self.prices          = ray.get(price_ref)
            self.window_size     = cfg.get('window_size', 30)
            self.initial_balance = cfg.get('initial_balance', 10_000.0)
            self.commission      = cfg.get('commission', 0.001)
            self.penalty_weight  = cfg.get('penalty_weight', 0.05)
            self.n_feat          = self.features.shape[1]
            n_obs = self.window_size * self.n_feat + 2
            self.observation_space = spaces.Box(
                low=-np.inf, high=np.inf, shape=(n_obs,), dtype=np.float32
            )
            self.action_space = spaces.Discrete(3)
            self._reset_state()

        def _reset_state(self):
            self.balance    = self.initial_balance
            self.position   = 0.0
            self.peak_value = self.initial_balance
            self.step_idx   = self.window_size
            self.trades     = []
            self.portfolio_history = []

        def _portfolio_value(self, price):
            return self.balance + self.position * price

        def _get_obs(self):
            window = self.features[self.step_idx - self.window_size: self.step_idx]
            price  = float(self.prices[self.step_idx])
            pv     = self._portfolio_value(price)
            port   = np.array([
                self.balance / self.initial_balance,
                self.position * price / self.initial_balance,
            ], dtype=np.float32)
            return np.concatenate([window.flatten(), port])

        def reset(self, *, seed=None, options=None):
            super().reset(seed=seed)
            self._reset_state()
            return self._get_obs(), {}

        def step(self, action):
            price      = float(self.prices[self.step_idx])
            prev_value = self._portfolio_value(price)
            if action == 1 and self.balance > 0:
                qty = self.balance * (1 - self.commission) / price
                self.position += qty
                self.balance   = 0.0
                self.trades.append({'step': self.step_idx, 'side': 'buy',  'price': price})
            elif action == 2 and self.position > 0:
                self.balance  += self.position * price * (1 - self.commission)
                self.position  = 0.0
                self.trades.append({'step': self.step_idx, 'side': 'sell', 'price': price})
            self.step_idx += 1
            done = self.step_idx >= len(self.prices) - 1
            new_price = float(self.prices[self.step_idx])
            new_value = self._portfolio_value(new_price)
            self.portfolio_history.append(new_value)
            step_ret = (new_value - prev_value) / (prev_value + 1e-8)
            if new_value > self.peak_value:
                self.peak_value = new_value
            drawdown = (self.peak_value - new_value) / (self.peak_value + 1e-8)
            reward   = float(step_ret - self.penalty_weight * drawdown)
            obs  = self._get_obs()
            info = {'portfolio_value': new_value, 'balance': self.balance,
                    'position': self.position, 'drawdown': drawdown,
                    'n_trades': len(self.trades)}
            return obs, reward, done, False, info

        def render(self):
            price = float(self.prices[self.step_idx])
            pv    = self._portfolio_value(price)
            dd    = (self.peak_value - pv) / (self.peak_value + 1e-8)
            print(f'step={self.step_idx:5d} | PV={pv:,.2f} | DD={dd:.2%} | trades={len(self.trades)}')
''')

ENV_FILE.write_text(env_source)
print(f'环境类已写入: {ENV_FILE}')

环境类已写入: /tmp/crypto_ppo_env.py


## 4. 数据集划分（8:2）

In [6]:
n_total = len(feat_matrix)
n_train = int(n_total * TRAIN_RATIO)
n_eval  = n_total - n_train

train_feat, eval_feat   = feat_matrix[:n_train], feat_matrix[n_train:]
train_price, eval_price = close_arr[:n_train],   close_arr[n_train:]
train_time  = feat['timestamp'].iloc[:n_train]
eval_time   = feat['timestamp'].iloc[n_train:]

print(f'总步数  : {n_total}')
print(f'训练集  : {n_train} 步  ({train_time.iloc[0]}  →  {train_time.iloc[-1]})')
print(f'验证集  : {n_eval} 步  ({eval_time.iloc[0]}  →  {eval_time.iloc[-1]})')

assert n_train > WINDOW_SIZE and n_eval > WINDOW_SIZE, \
    f'数据集太短，请缩小 WINDOW_SIZE 或扩大日期范围'
print('✓ 数据集检查通过')

总步数  : 5741
训练集  : 4592 步  (2026-04-09 00:19:00+08:00  →  2026-04-12 04:50:00+08:00)
验证集  : 1149 步  (2026-04-12 04:51:00+08:00  →  2026-04-12 23:59:00+08:00)
✓ 数据集检查通过


## 5. RLlib PPO 并行训练

### 架构
```
┌─────────────────────────────────────────┐
│  Ray Cluster (本机 10 核)                │
│                                         │
│  ┌──────────┐   rollout   ┌──────────┐  │
│  │ Learner  │ ◄────────── │ Worker×9 │  │
│  │ (1 核)   │  batches    │ (9 核)   │  │
│  └──────────┘             └──────────┘  │
└─────────────────────────────────────────┘
```

- **Worker** 并行采集经验（9 进程同时运行环境）
- **Learner** 汇总 batch 并更新网络权重
- 大数组（feature matrix）通过 `ray.put()` 共享，避免序列化复制

In [7]:
import ray
from ray.rllib.algorithms.ppo import PPOConfig
from ray import tune
from pathlib import Path

Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)
Path(LOG_DIR).mkdir(parents=True, exist_ok=True)

# 初始化 Ray（使用 10 核）
# runtime_env 中设置 PYTHONPATH，让 worker 子进程能 import /tmp/crypto_ppo_env
if ray.is_initialized():
    ray.shutdown()
ray.init(
    num_cpus=10,
    ignore_reinit_error=True,
    runtime_env={'env_vars': {'PYTHONPATH': '/tmp'}},
)
print(f'Ray 已初始化: {ray.cluster_resources()}')

# 将大数组放入 Ray 对象存储（零拷贝共享给所有 worker）
train_feat_ref  = ray.put(train_feat)
train_price_ref = ray.put(train_price)

# 注册环境（主进程也需要能 import，用于 build() 时的本地验证）
import sys
if '/tmp' not in sys.path:
    sys.path.insert(0, '/tmp')
from crypto_ppo_env import CryptoPPOEnv as _CryptoPPOEnv

tune.register_env('CryptoPPOEnv', lambda cfg: _CryptoPPOEnv(cfg))

env_config = {
    'features_ref':    train_feat_ref,
    'prices_ref':      train_price_ref,
    'window_size':     WINDOW_SIZE,
    'initial_balance': INITIAL_BALANCE,
    'commission':      COMMISSION,
    'penalty_weight':  PENALTY_WEIGHT,
}

# 构建 PPO 配置
config = (
    PPOConfig()
    .environment(env='CryptoPPOEnv', env_config=env_config)
    .env_runners(
        num_env_runners=NUM_WORKERS,
        num_cpus_per_env_runner=1,
        rollout_fragment_length='auto',
    )
    .learners(
        num_learners=NUM_LEARNERS,
        num_cpus_per_learner=1,
    )
    .training(
        lr=PPO_KWARGS['lr'],
        train_batch_size_per_learner=PPO_KWARGS['train_batch_size_per_learner'],
        minibatch_size=PPO_KWARGS['minibatch_size'],
        num_epochs=PPO_KWARGS['num_epochs'],
        gamma=PPO_KWARGS['gamma'],
        lambda_=PPO_KWARGS['lambda_'],
        clip_param=PPO_KWARGS['clip_param'],
        entropy_coeff=PPO_KWARGS['entropy_coeff'],
        vf_loss_coeff=PPO_KWARGS['vf_loss_coeff'],
        grad_clip=PPO_KWARGS['grad_clip'],
    )
    .framework('torch')
    .debugging(log_level='WARN')
)

algo = config.build()
print(f'PPO 算法已构建')
n_obs = WINDOW_SIZE * len(FEATURE_COLS) + 2
print(f'观测维度 : ({n_obs},)')
print(f'目标步数 : {TOTAL_TIMESTEPS:,}')
print(f'Workers  : {NUM_WORKERS} rollout + {NUM_LEARNERS} learner')

/home/davidlyu/projects/GinkgoBrain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-14 18:42:45,134	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-14 18:42:45,490	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-04-14 18:42:48,307	INFO worker.py:2013 -- Started a local Ray instance.
/home/davidlyu/projects/GinkgoBrain/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error messag

Ray 已初始化: {'accelerator_type:G': 1.0, 'memory': 6528909722.0, 'node:172.20.39.187': 1.0, 'CPU': 10.0, 'object_store_memory': 2798104166.0, 'GPU': 1.0, 'node:__internal_head__': 1.0}


(SingleAgentEnvRunner pid=832521) DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
2026-04-14 18:42:56,788	WARNING rl_module.py:463 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
2026-04-14 18:42:56,859	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
(_WrappedExecutable pid=833068) Setting up process group for: env:// [rank=0, world_size=1]


(_WrappedExecutable pid=833068) [Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


2026-04-14 18:43:01,821	INFO trainable.py:161 -- Trainable.setup took 12.673 seconds. If your trainable is slow to initialize, consider setting reuse_actors=True to reduce actor creation overheads.
2026-04-14 18:43:01,822	WARNING util.py:62 -- Install gputil for GPU system monitoring.


PPO 算法已构建
观测维度 : (332,)
目标步数 : 1,000,000
Workers  : 9 rollout + 1 learner


In [8]:
# ── 训练循环（计时）────────────────────────────────────────────────
from tqdm.auto import tqdm

train_rewards   = []
train_steps_log = []
total_steps     = 0
iteration       = 0

train_batch_size = PPO_KWARGS['train_batch_size_per_learner'] * NUM_LEARNERS
max_iterations   = (TOTAL_TIMESTEPS // train_batch_size) + 1

print(f'每次迭代 batch 大小: {train_batch_size} 步')
print(f'预计迭代次数       : {max_iterations}')
print('开始训练...')

t0 = time.time()
best_reward = -np.inf
best_ckpt   = None

pbar = tqdm(total=TOTAL_TIMESTEPS, desc='RLlib PPO', unit='step', dynamic_ncols=True)

while total_steps < TOTAL_TIMESTEPS:
    result      = algo.train()
    iteration  += 1
    prev_steps  = total_steps
    total_steps = result.get('num_env_steps_sampled_lifetime', total_steps)
    ep_reward   = result.get('env_runners', {}).get('episode_reward_mean', float('nan'))
    train_rewards.append(ep_reward)
    train_steps_log.append(total_steps)

    pbar.update(total_steps - prev_steps)
    reward_str = f'{ep_reward:.4f}' if not np.isnan(ep_reward) else 'nan'
    pbar.set_postfix({'iter': iteration, 'reward': reward_str,
                      'elapsed': f'{time.time()-t0:.0f}s'})

    # 保存最优 checkpoint
    if not np.isnan(ep_reward) and ep_reward > best_reward:
        best_reward = ep_reward
        best_ckpt = algo.save_to_path(str(__import__('pathlib').Path(f'{MODEL_DIR}/{RUN_NAME}/best').resolve()))
    elif best_ckpt is None and not np.isnan(ep_reward):
        best_reward = ep_reward
        best_ckpt = algo.save_to_path(str(__import__('pathlib').Path(f'{MODEL_DIR}/{RUN_NAME}/best').resolve()))

pbar.close()
elapsed_total = time.time() - t0

# 保存最终模型
final_ckpt = algo.save_to_path(str(__import__('pathlib').Path(f'{MODEL_DIR}/{RUN_NAME}/final').resolve()))

sb3_sps   = SB3_BASELINE_STEPS / SB3_BASELINE_SECONDS
rllib_sps = total_steps / elapsed_total
speedup   = rllib_sps / sb3_sps

print()
print('=' * 50)
print(f'  训练完成')
print(f'  总步数     : {total_steps:,}')
print(f'  总耗时     : {elapsed_total:.1f} 秒  ({elapsed_total/60:.1f} 分钟)')
print(f'  平均速度   : {rllib_sps:.0f} steps/s')
print(f'  最佳奖励   : {best_reward:.4f}')
print(f'  SB3 基准   : {SB3_BASELINE_SECONDS} 秒 for {SB3_BASELINE_STEPS:,} 步  ({sb3_sps:.0f} steps/s)')
print(f'  速度提升   : {speedup:.1f}x')
print('=' * 50)
print(f'最终 checkpoint: {final_ckpt}')


每次迭代 batch 大小: 2048 步
预计迭代次数       : 489
开始训练...


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## 6. 回测评估

加载最佳 checkpoint，在验证集上运行确定性策略，计算收益、回撤、夏普比率。

In [ ]:
# ── 用已训练的 algo 直接推理，无需 from_checkpoint，无 Ray 通信开销 ──
import torch

rl_module = algo.get_module()
rl_module.eval()

def get_action(obs_np: np.ndarray) -> int:
    with torch.no_grad():
        obs_t  = torch.FloatTensor(obs_np).unsqueeze(0)
        out    = rl_module.forward_inference({'obs': obs_t})
        return int(torch.argmax(out['action_dist_inputs'], dim=-1).item())

# ── 纯 numpy 内联回测（不使用 Ray worker，不调用 CryptoPPOEnv）────
balance    = INITIAL_BALANCE
position   = 0.0
peak_value = INITIAL_BALANCE
step_idx   = WINDOW_SIZE
portfolio_history: list[float] = []
trades:            list[dict]  = []

def _get_obs(si):
    window = eval_feat[si - WINDOW_SIZE : si].flatten()
    price  = float(eval_price[si])
    pv     = balance + position * price
    return np.concatenate([
        window,
        np.array([balance / INITIAL_BALANCE,
                  position * price / INITIAL_BALANCE], dtype=np.float32)
    ])

obs = _get_obs(step_idx)

t_eval0 = time.time()
while step_idx < len(eval_price) - 1:
    action = get_action(obs)
    price  = float(eval_price[step_idx])

    if action == 1 and balance > 0:
        qty       = balance * (1 - COMMISSION) / price
        position += qty
        balance   = 0.0
        trades.append({'step': step_idx, 'side': 'buy',  'price': price})
    elif action == 2 and position > 0:
        balance  += position * price * (1 - COMMISSION)
        position  = 0.0
        trades.append({'step': step_idx, 'side': 'sell', 'price': price})

    step_idx += 1
    new_price = float(eval_price[step_idx])
    pv        = balance + position * new_price
    portfolio_history.append(pv)
    if pv > peak_value:
        peak_value = pv

    obs = _get_obs(step_idx)

print(f'回测完成，耗时 {time.time()-t_eval0:.1f}s，共 {len(portfolio_history)} 步')

pv_curve    = np.array(portfolio_history)
prices_eval = eval_price[WINDOW_SIZE + 1:]

# ── 指标 ─────────────────────────────────────────────────────────
total_return = pv_curve[-1] / INITIAL_BALANCE - 1
peak         = np.maximum.accumulate(pv_curve)
drawdowns    = (peak - pv_curve) / (peak + 1e-8)
max_dd       = drawdowns.max()
step_rets    = np.diff(pv_curve) / (pv_curve[:-1] + 1e-8)
sharpe       = (step_rets.mean() / (step_rets.std() + 1e-8)) * np.sqrt(365 * 24 * 60)
bh_return    = eval_price[-1] / eval_price[WINDOW_SIZE] - 1

print(f"{'='*40}")
print(f'  验证集回测结果 (RLlib PPO)')
print(f"{'='*40}")
print(f'  总收益率   : {total_return:+.2%}')
print(f'  买入持有   : {bh_return:+.2%}  (baseline)')
print(f'  最大回撤   : {max_dd:.2%}')
print(f'  夏普比率   : {sharpe:.2f}')
print(f'  交易次数   : {len(trades)}')
print(f"{'='*40}")


## 7. 可视化

In [ ]:
ts_eval = eval_time.iloc[WINDOW_SIZE + 1:].reset_index(drop=True)

fig = plt.figure(figsize=(16, 14))
gs  = gridspec.GridSpec(4, 2, figure=fig, hspace=0.45, wspace=0.3)
fig.suptitle(f'{SYMBOL}  RLlib PPO 训练结果', fontsize=14)

# ── (上左) 训练奖励曲线 ────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
valid_mask = [not np.isnan(r) for r in train_rewards]
valid_steps   = [train_steps_log[i] for i in range(len(train_steps_log)) if valid_mask[i]]
valid_rewards = [train_rewards[i]    for i in range(len(train_rewards))   if valid_mask[i]]
ax0.plot(valid_steps, valid_rewards, color='#2196f3', linewidth=1)
if len(valid_rewards) > 20:
    smooth = pd.Series(valid_rewards).rolling(20, min_periods=1).mean()
    ax0.plot(valid_steps, smooth, color='#ff9800', linewidth=1.5, label='平滑 (20步)')
    ax0.legend(fontsize=8)
ax0.set_xlabel('训练步数')
ax0.set_ylabel('平均 Episode 奖励')
ax0.set_title('训练奖励曲线')
ax0.grid(True, alpha=0.3)

# ── (上右) SB3 vs RLlib 速度对比 ────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
labels    = ['SB3 (1核)', f'RLlib ({NUM_WORKERS+NUM_LEARNERS}核)']
sps_vals  = [sb3_sps, rllib_sps]
colors    = ['#ff9800', '#4caf50']
bars = ax1.bar(labels, sps_vals, color=colors, width=0.4, edgecolor='white')
for bar, val in zip(bars, sps_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{val:.0f}', ha='center', va='bottom', fontsize=10)
ax1.set_ylabel('Steps / 秒')
ax1.set_title(f'训练速度对比  ({speedup:.1f}x 提升)')
ax1.grid(True, alpha=0.3, axis='y')

# ── (中) 资产曲线 ────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1, :])
bh_curve = INITIAL_BALANCE * (prices_eval / prices_eval[0])
ax2.plot(ts_eval, pv_curve,  label='RLlib PPO', color='#2196f3', linewidth=1.2)
ax2.plot(ts_eval, bh_curve,  label='买入持有',  color='#ff9800', linewidth=1, linestyle='--', alpha=0.8)
ax2.fill_between(ts_eval, pv_curve, bh_curve,
                 where=pv_curve >= bh_curve, alpha=0.15, color='#4caf50', label='策略超额')
ax2.fill_between(ts_eval, pv_curve, bh_curve,
                 where=pv_curve < bh_curve,  alpha=0.15, color='#ef5350')
buy_steps  = [t['step'] - WINDOW_SIZE for t in trades if t['side'] == 'buy'  and t['step'] - WINDOW_SIZE < len(ts_eval)]
sell_steps = [t['step'] - WINDOW_SIZE for t in trades if t['side'] == 'sell' and t['step'] - WINDOW_SIZE < len(ts_eval)]
if buy_steps:
    ax2.scatter(ts_eval.iloc[buy_steps],  pv_curve[buy_steps],
                color='#4caf50', marker='^', s=30, zorder=5, label='买入')
if sell_steps:
    ax2.scatter(ts_eval.iloc[sell_steps], pv_curve[sell_steps],
                color='#ef5350', marker='v', s=30, zorder=5, label='卖出')
ax2.set_ylabel('资产价值 (USD)')
ax2.set_title('验证集资产曲线')
ax2.legend(loc='upper left', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.tick_params(axis='x', rotation=30)

# ── (下左) 回撤曲线 ─────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2, 0])
ax3.fill_between(ts_eval, drawdowns * 100, alpha=0.6, color='#ef5350')
ax3.axhline(max_dd * 100, color='red', linestyle='--', linewidth=1,
            label=f'最大回撤 {max_dd:.2%}')
ax3.set_ylabel('回撤 (%)')
ax3.set_title('回撤曲线')
ax3.invert_yaxis()
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.tick_params(axis='x', rotation=30)

# ── (下右) 滚动夏普 ─────────────────────────────────────────────
ax4 = fig.add_subplot(gs[2, 1])
roll_w = 60
roll_ret    = pd.Series(step_rets)
roll_sharpe = (roll_ret.rolling(roll_w).mean()
               / (roll_ret.rolling(roll_w).std() + 1e-8)) * np.sqrt(365 * 24 * 60)
ax4.plot(ts_eval.iloc[1:], roll_sharpe, color='#ba68c8', linewidth=1)
ax4.axhline(0,  color='gray',    linestyle='--', linewidth=0.8, alpha=0.5)
ax4.axhline(1,  color='#4caf50', linestyle='--', linewidth=0.8, alpha=0.7)
ax4.axhline(-1, color='#ef5350', linestyle='--', linewidth=0.8, alpha=0.7)
ax4.set_ylabel(f'滚动夏普 ({roll_w}步)')
ax4.set_title('滚动夏普比率')
ax4.grid(True, alpha=0.3)
ax4.tick_params(axis='x', rotation=30)

# ── (底部) 速度比较汇总文本 ──────────────────────────────────────
ax5 = fig.add_subplot(gs[3, :])
ax5.axis('off')
summary = (
    f'训练速度汇总\n'
    f'SB3 (1核):  {SB3_BASELINE_STEPS:,} 步 / {SB3_BASELINE_SECONDS}s = {sb3_sps:.0f} steps/s\n'
    f'RLlib ({NUM_WORKERS+NUM_LEARNERS}核): {total_steps:,} 步 / {elapsed_total:.0f}s = {rllib_sps:.0f} steps/s   '
    f'→ 速度提升 {speedup:.1f}x\n'
    f'验证集回测: 总收益 {total_return:+.2%} | 最大回撤 {max_dd:.2%} | 夏普 {sharpe:.2f} | 交易 {len(trades)} 次'
)
ax5.text(0.5, 0.5, summary, ha='center', va='center', fontsize=11,
         transform=ax5.transAxes,
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#1e1e2e', edgecolor='#4caf50', linewidth=1.5))

plt.savefig(f'{MODEL_DIR}/{RUN_NAME}_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('图表已保存')